In [0]:
from pyspark.sql.functions import *

In [0]:
# Dados de Silver
CATALOG_SILVER = "silver"
SCHEMA_SILVER = "vendas"
TABLE_SILVER = "produtos"

# Dados de Gold
CATALOG_GOLD = "gold"
SCHEMA_GOLD = "vendas"
TABLE_GOLD = "dim_clientes"

In [0]:
print(f"CATALOG_SILVER: {CATALOG_SILVER}")
print(f"SCHEMA_SILVER: {SCHEMA_SILVER}")
print(f"TABLE_SILVER: {TABLE_SILVER}")
print("--------------------------")
print(f"CATALOG_GOLD: {CATALOG_GOLD}")
print(f"SCHEMA_GOLD: {SCHEMA_GOLD}")
print(f"TABLE_GOLD: {TABLE_GOLD}")

In [0]:
# Cria schema vendas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_GOLD}.{SCHEMA_GOLD}")

In [0]:
df_clientes = spark.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.{TABLE_SILVER}").select(
    "id_cliente",
    "email_nome",
    "codigo_postal",
    "cidade",
    "estado",
    "regiao",
    "distrito",
    "pais",
)
df_clientes = df_clientes.dropDuplicates()

In [0]:
df_clientes = df_clientes.withColumn("email", lower(regexp_replace(split(col("email_nome"), ":").getItem(0), r"[()]", "")))


In [0]:
df_clientes = df_clientes.withColumn("nome", split(col("email_nome"), ",").getItem(1))

In [0]:
df_clientes = df_clientes.withColumn("sobrenome", split(split(col("email_nome"), ",").getItem(0), ":").getItem(1))


In [0]:
df_clientes = df_clientes.withColumn("cidade", split(col("cidade"), ",").getItem(0))

In [0]:
df_clientes = df_clientes.select("id_cliente", "nome", "sobrenome", "email", "codigo_postal", "cidade", "estado", "regiao", "distrito", "pais")

In [0]:
df_clientes = df_clientes.orderBy("id_cliente")

In [0]:
df_clientes.write.mode("overwrite").saveAsTable(f"{CATALOG_GOLD}.{SCHEMA_GOLD}.{TABLE_GOLD}")
